# 07 — Failing Plan Analysis

Compares `FlanT5_Stepwise_FT_Combined` (proposed method) and `G5_stepwise` (GPT-5)
on how they handle invalid plans.

## Structure

**Cell 1:** Among all gold invalid plans — how many correctly predicted invalid vs missed.

**Cell 2:** Among correctly predicted invalid plans — confusion matrix by fail type:
- Rows: gold fail type (action-fail / goal-fail)
- Columns: predicted fail type (action-fail = model fired B / goal-fail = model passed all steps, goal check caught it)

## Prerequisites
Run notebooks 02 and 05 first.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, json

ROOT        = '/content/drive/MyDrive/paper_codes/paper2'
RESULTS_DIR = f'{ROOT}/results'
DATA_DIR    = f'{ROOT}/data'

DOMAINS = ['blocksworld', 'blocksworld_3', 'mystery', 'mystery_3', 'logistics']
CONDS   = ['FlanT5_Stepwise_FT_Combined', 'G5_stepwise']

# Load raw predictions
raw_all = {}
for fname in ['raw_preds_flant5.json', 'raw_preds_gpt5.json']:
    path = os.path.join(RESULTS_DIR, fname)
    if os.path.exists(path):
        raw_all.update(json.load(open(path)))
        print(f'Loaded: {fname}')
    else:
        print(f'MISSING: {fname}')

def get_arrays(cond, domain):
    r   = raw_all.get(cond, {}).get(domain, {})
    pg  = r.get('plan_golds',   [])
    pp  = r.get('plan_preds',   [])
    fsg = r.get('fail_step_golds') or r.get('failing_step_golds') or []
    fsp = r.get('fail_step_preds') or r.get('failing_step_preds') or []
    return pg, pp, fsg, fsp

# Build plan_id → fail_reason from stepwise test file
# Each row in the stepwise test file has plan_id, plan_label, fail_reason
# We only need one row per plan (all steps of a plan share the same fail_reason)
plan_fail_reason = {}  # (domain, plan_id) -> fail_reason
sw_test_path = os.path.join(DATA_DIR, 'planbench_stepwise_test.jsonl')
with open(sw_test_path) as f:
    for line in f:
        r = json.loads(line)
        key = (r['domain'], r['plan_id'])
        if key not in plan_fail_reason:
            plan_fail_reason[key] = r.get('fail_reason', 'unknown')

print(f'\nLoaded fail_reason for {len(plan_fail_reason)} (domain, plan_id) pairs')

# Also build plan_id → order index so we can align plan_golds with plan_ids
# We need the ordered list of plan_ids per domain per condition
# The plan_golds array follows the same order as plan_ids sorted in evaluate_stepwise
# Rebuild from sw_test: sorted plan_ids per domain
from collections import defaultdict
sw_plans_by_domain = defaultdict(set)
with open(sw_test_path) as f:
    for line in f:
        r = json.loads(line)
        sw_plans_by_domain[r['domain']].add(r['plan_id'])

sorted_plan_ids = {d: sorted(sw_plans_by_domain[d]) for d in DOMAINS}
print('Plan counts per domain:', {d: len(v) for d, v in sorted_plan_ids.items()})

## Cell 1 — Gold Invalid Plans: Predicted Invalid vs Missed

Among all gold invalid plans, how many does each condition correctly predict as invalid
(pred=0) vs completely miss (pred=1 / valid)?

In [ ]:
SEP = '=' * 70

print(SEP)
print('GOLD INVALID PLANS — PREDICTED INVALID vs MISSED')
print(SEP)

for cond in CONDS:
    print(f'\nCondition: {cond}')
    print(f'  {"Domain":<15s}  {"Gold Invalid":>13s}  {"Pred Invalid":>13s}  {"Missed":>7s}  {"Miss%":>7s}')
    print('  ' + '-' * 60)

    t_gi = t_pi = t_fn = 0
    for domain in DOMAINS:
        pg, pp, fsg, fsp = get_arrays(cond, domain)
        if not pg: continue
        n_gi = sum(1 for g in pg if g == 0)
        n_pi = sum(1 for g,p in zip(pg,pp) if g==0 and p==0)
        n_fn = sum(1 for g,p in zip(pg,pp) if g==0 and p==1)
        miss_pct = n_fn / max(1, n_gi) * 100
        t_gi += n_gi; t_pi += n_pi; t_fn += n_fn
        print(f'  {domain:<15s}  {n_gi:>13d}  {n_pi:>13d}  {n_fn:>7d}  {miss_pct:>6.1f}%')

    t_miss_pct = t_fn / max(1, t_gi) * 100
    print('  ' + '-' * 60)
    print(f'  {"OVERALL":<15s}  {t_gi:>13d}  {t_pi:>13d}  {t_fn:>7d}  {t_miss_pct:>6.1f}%')

## Cell 2 — Correctly Predicted Invalid Plans: Confusion Matrix by Fail Type

Only considers plans where **gold=invalid AND pred=invalid** (correctly caught plans).

**Rows = gold fail type:**
- `action-fail`: plan fails because an action has an unmet precondition
- `goal-fail`: all actions are valid but goal state is not reached

**Columns = predicted fail type:**
- `pred action-fail`: model fired B at some step (stopped pipeline early)
- `pred goal-fail`: model predicted all steps as A, goal check returned invalid

This reveals whether each condition correctly identifies **why** a plan fails,
not just that it fails.

In [ ]:
print(SEP)
print('CONFUSION MATRIX BY FAIL TYPE')
print('Only correctly predicted invalid plans (gold=0, pred=0)')
print(SEP)
print()
print('Rows    = gold fail type  (action-fail / goal-fail)')
print('Columns = pred fail type  (pred action-fail / pred goal-fail)')
print()

for cond in CONDS:
    print(f'{'─'*70}')
    print(f'CONDITION: {cond}')
    print()

    # For each domain, align plan_golds/preds with plan_ids
    # then look up fail_reason per plan
    # pred_fail_type: if plan_id is in fsp (has pred_fail_step) -> action-fail
    #                 else -> goal-fail (reached goal check)

    # We need plan_id order — same as sorted_plan_ids[domain]
    # plan_golds is in that order from evaluate_stepwise

    # Build set of plan_ids where model predicted action-fail
    # = plan_ids that appear in fail_step_preds alignment
    # fsg/fsp are aligned: fsg[i] = gold_fs, fsp[i] = pred_fs for plan i
    # But we don't have plan_ids in fsg/fsp — only the step numbers
    #
    # Alternative: pred_action_fail plan_ids = plans where pred=0 AND
    # the model stopped before exhausting all steps = has pred_fail_step
    # We can derive this: for each domain, plans where pred=0 are action-fail
    # predictions UNLESS the model passed all steps and goal check failed.
    # The fail_step_preds array tells us which invalid-predicted plans fired B.
    # len(fsp) = number of plans that fired B (pred action-fail AND gold action-fail)
    # But we also need pred action-fail on gold goal-fail plans.
    #
    # Key insight from evaluate_stepwise logic:
    # - If pred == 0: plan_pred = 0, stopped = True -> pred_fail_type = action-fail
    # - If all steps A and goal_check returns 0: plan_pred = 0, stopped = False -> pred_fail_type = goal-fail
    #
    # We cannot distinguish these two cases from plan_golds/preds alone.
    # BUT: we know that fsg/fsp length = plans where BOTH:
    #   (a) gold has a B step (action-fail plan)
    #   (b) model predicted invalid (pred=0)
    # This is a SUBSET of pred action-fail plans.
    #
    # Best approximation with available data:
    # pred_action_fail = plans where pred=0 AND gold is action-fail (from fsg alignment)
    #                  PLUS plans where pred=0 AND gold is goal-fail (model fired B on goal-fail plan)
    # pred_goal_fail   = plans where pred=0 AND goal check was needed
    #
    # Since we have plan_id order and fail_reason lookup, we can do this properly:
    # For each correctly-predicted-invalid plan:
    #   gold_fail_type = fail_reason from test file
    #   pred_fail_type = action-fail if plan_id in fsp_plan_ids else goal-fail
    #
    # fsp_plan_ids: reconstruct from fsg/fsp and the plan_id order
    # fsg are step numbers of gold failing steps for plans where model fired B
    # These correspond to: invalid plans (gold=0) where pred=0, in plan_id order
    # So we can reconstruct fsp_plan_ids by iterating sorted_plan_ids[domain]
    # and counting only gold=0 and pred=0 plans

    # Per-domain matrices, then aggregate
    # 2x2: rows=gold(action,goal), cols=pred(action,goal)
    total_matrix = [[0,0],[0,0]]  # [gold_action][pred_action], [gold_action][pred_goal]
                                   # [gold_goal][pred_action],   [gold_goal][pred_goal]

    for domain in DOMAINS:
        pg, pp, fsg, fsp = get_arrays(cond, domain)
        if not pg: continue

        pids = sorted_plan_ids[domain]
        if len(pids) != len(pg):
            print(f'  WARNING: {domain} plan count mismatch: {len(pids)} vs {len(pg)}')
            continue

        # Build fsp_plan_ids: plan_ids that fired B (pred action-fail)
        # These are the plans counted in fsp, in order of pids where g==0 and p==0
        fsp_idx = 0
        fsp_plan_ids = set()
        for pid, g, p in zip(pids, pg, pp):
            if g == 0 and p == 0:  # correctly caught invalid plan
                if fsp_idx < len(fsp):
                    # This plan has a pred_fail_step recorded -> fired B
                    fsp_plan_ids.add(pid)
                    fsp_idx += 1

        # Build confusion matrix for this domain
        for pid, g, p in zip(pids, pg, pp):
            if not (g == 0 and p == 0):  # only correctly predicted invalid
                continue
            gold_fail = plan_fail_reason.get((domain, pid), 'unknown')
            pred_fired_b = pid in fsp_plan_ids

            gold_row = 0 if gold_fail == 'action_fail' else 1  # 0=action, 1=goal
            pred_col = 0 if pred_fired_b else 1                 # 0=action, 1=goal
            total_matrix[gold_row][pred_col] += 1

    # Print confusion matrix
    aa = total_matrix[0][0]  # gold action-fail, pred action-fail
    ag = total_matrix[0][1]  # gold action-fail, pred goal-fail
    ga = total_matrix[1][0]  # gold goal-fail,   pred action-fail
    gg = total_matrix[1][1]  # gold goal-fail,   pred goal-fail

    total = aa + ag + ga + gg
    print(f'  (All domains combined, n={total} correctly predicted invalid plans)')
    print()
    print(f'  {"":25s}  {"Pred: Action-fail":>18s}  {"Pred: Goal-fail":>16s}  {"Total":>7s}')
    print('  ' + '-' * 72)
    print(f'  {"Gold: Action-fail":<25s}  {aa:>18d}  {ag:>16d}  {aa+ag:>7d}')
    print(f'  {"Gold: Goal-fail":<25s}  {ga:>18d}  {gg:>16d}  {ga+gg:>7d}')
    print('  ' + '-' * 72)
    print(f'  {"Total":<25s}  {aa+ga:>18d}  {ag+gg:>16d}  {total:>7d}')
    print()

    # Key ratios
    n_gold_action = aa + ag
    n_gold_goal   = ga + gg
    if n_gold_action > 0:
        print(f'  Of {n_gold_action} correctly-caught action-fail plans:')
        print(f'    {aa} ({aa/n_gold_action*100:.1f}%) predicted as action-fail (model fired B)')
        print(f'    {ag} ({ag/n_gold_action*100:.1f}%) predicted as goal-fail  (model passed all steps, goal check caught it)')
    if n_gold_goal > 0:
        print(f'  Of {n_gold_goal} correctly-caught goal-fail plans:')
        print(f'    {gg} ({gg/n_gold_goal*100:.1f}%) predicted as goal-fail  (correct)')
        print(f'    {ga} ({ga/n_gold_goal*100:.1f}%) predicted as action-fail (model fired B on a valid step)')
    print()